# Video 07 - Exercise Solutions

Try the exercises in [exercises.md](../exercises.md) yourself first. These are **one** good answer each.

In [0]:
import json, random
import pandas as pd
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"

random.seed(11)   # same fake data as the main notebook
ORDERS = [
    {"order_id": 1000 + i, "customer_id": random.choice([7, 42, 99]),
     "date": f"2026-{random.randint(1, 6):02d}-{random.randint(1, 28):02d}",
     "status": random.choice(["delivered", "delivered", "shipped", "cancelled"]), "items": random.randint(1, 5),
     "total": round(random.uniform(10, 400), 2), "shipping_address": "12 Example Street, Perth WA 6000",
     "notes": "Leave at front door if nobody is home."}
    for i in range(400)
]

def add_tool_results(messages, msg, functions):
    messages.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        result = functions[call.function.name](**json.loads(call.function.arguments or "{}"))
        content = result if isinstance(result, str) else json.dumps(result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

def ask_with_tools(question, tools, functions, model=MODEL):
    messages = [{"role": "user", "content": question}]
    msg = client.chat.completions.create(model=model, messages=messages, tools=tools).choices[0].message
    if not msg.tool_calls:
        return msg.content, 0
    add_tool_results(messages, msg, functions)
    response = client.chat.completions.create(model=model, messages=messages, tools=tools)
    return response.choices[0].message.content, response.usage.prompt_tokens

In [0]:
def check_tool_messages(messages):
    """Return a list of golden-rule violations in a message list (empty list = all good)."""
    problems, requested, answered = [], set(), set()
    for i, m in enumerate(messages):
        for call in m.get("tool_calls") or []:
            requested.add(call["id"])
        if m.get("role") == "tool":
            if m.get("tool_call_id") not in requested:
                problems.append(f"message {i}: tool result for unknown id {m.get('tool_call_id')!r} (rules 1-2)")
            if not isinstance(m.get("content"), str):
                problems.append(f"message {i}: content is {type(m.get('content')).__name__}, not str (rule 3)")
            answered.add(m.get("tool_call_id"))
    for missing in requested - answered:
        problems.append(f"tool call {missing!r} was never answered (rule 4)")
    return problems

req = {"role": "assistant", "tool_calls": [{"id": "c1", "type": "function", "function": {"name": "get_weather", "arguments": "{}"}},
                                          {"id": "c2", "type": "function", "function": {"name": "get_weather", "arguments": "{}"}}]}
q = {"role": "user", "content": "Weather in Perth and Tokyo?"}
tests = {
    "correct": [q, req, {"role": "tool", "tool_call_id": "c1", "content": "sunny"}, {"role": "tool", "tool_call_id": "c2", "content": "cloudy"}],
    "no request": [q, {"role": "tool", "tool_call_id": "c1", "content": "sunny"}],
    "wrong id": [q, req, {"role": "tool", "tool_call_id": "zzz", "content": "sunny"}, {"role": "tool", "tool_call_id": "c2", "content": "cloudy"}],
    "dict content": [q, req, {"role": "tool", "tool_call_id": "c1", "content": {"sky": "sunny"}}, {"role": "tool", "tool_call_id": "c2", "content": "cloudy"}],
    "one unanswered": [q, req, {"role": "tool", "tool_call_id": "c1", "content": "sunny"}],
}
for name, msgs in tests.items():
    print(f"{name:15} -> {check_tool_messages(msgs) or 'OK'}")

In [0]:
def get_top_customers(month: str, n: int = 3):
    totals = {}
    for o in ORDERS:
        if o["date"].startswith(month) and o["status"] != "cancelled":
            totals[o["customer_id"]] = totals.get(o["customer_id"], 0) + o["total"]
    top = sorted(totals.items(), key=lambda kv: kv[1], reverse=True)[:n]
    return [{"customer_id": c, "total": round(t, 2)} for c, t in top]

TOP_TOOL = {"type": "function", "function": {
    "name": "get_top_customers", "description": "Top customers by spend for one month, excluding cancelled orders.",
    "parameters": {"type": "object", "properties": {
        "month": {"type": "string", "description": "YYYY-MM"}, "n": {"type": "integer"}}, "required": ["month", "n"]}}}

answer, tokens = ask_with_tools("Which customer spent the most in April 2026?", [TOP_TOOL], {"get_top_customers": get_top_customers})
print(f"input tokens: {tokens:,}\n{answer}")

df = pd.DataFrame(ORDERS)
check = df[(df.date.str.startswith("2026-04")) & (df.status != "cancelled")].groupby("customer_id").total.sum().sort_values(ascending=False)
print("\npandas check:\n", check.round(2))

In [0]:
KEEP = ["order_id", "customer_id", "date", "status", "total"]

def get_all_orders_trimmed():
    return [{k: o[k] for k in KEEP} for o in ORDERS]

TRIMMED_TOOL = {"type": "function", "function": {
    "name": "get_all_orders", "description": "Return every order (id, customer, date, status, total).",
    "parameters": {"type": "object", "properties": {}}}}

answer, tokens = ask_with_tools("How much did customer 42 spend in March 2026, not counting cancelled orders?",
                                [TRIMMED_TOOL], {"get_all_orders": get_all_orders_trimmed})
truth = round(sum(o["total"] for o in ORDERS if o["customer_id"] == 42 and o["date"].startswith("2026-03") and o["status"] != "cancelled"), 2)
print(f"input tokens: {tokens:,} (the full dump was about 31,000)")
print(f"true total: ${truth} | model: ...{answer[-80:]!r}")
# In our test trimming roughly halved the input (about 17,000 tokens instead of 31,000)
# - but Llama still got the total wrong ($1,082.25), because it still has to add numbers in its head.
# The narrow tool (do the maths in code) is the reliable design.

In [0]:
def get_order_total(customer_id: int, month: str):
    if customer_id not in {o["customer_id"] for o in ORDERS}:
        return {"error": f"unknown customer {customer_id}"}          # an error, returned as a normal result
    rows = [o for o in ORDERS if o["customer_id"] == customer_id and o["date"].startswith(month) and o["status"] != "cancelled"]
    return {"customer_id": customer_id, "month": month, "total": round(sum(o["total"] for o in rows), 2)}

NARROW_TOOL = {"type": "function", "function": {
    "name": "get_order_total", "description": "Total spent by one customer in one month, excluding cancelled orders.",
    "parameters": {"type": "object", "properties": {"customer_id": {"type": "integer"}, "month": {"type": "string"}},
                   "required": ["customer_id", "month"]}}}

answer, _ = ask_with_tools("How much did customer 12345 spend in March 2026?", [NARROW_TOOL], {"get_order_total": get_order_total})
print(answer)
# Returning the error as a result lets the model explain the problem to the user (or try something else).
# A Python crash would just stop everything. Video 09 builds on this.